# Notes from AIMA Chapter 6 Constraint Satisfaction Problems

- Constraint satisfaction problem: break open the black box (atomic states with no internal structure) by using a **factored representation** for each state:
    - a set of variables, each has a value
    - problem solved when each variable has a value that satisfies all the constraints on the variable

## Defining CSPs

- Assignment of values to variables that does not violate any constraints = **consistent**
- **complete assignment** - every var assigned a value
- **solutions** = consistent and complete
- **partial assignment** = some vars unassigned
- **partial solution** = partial assignment that is consistent
- CSPs are NP-complete problems
- **constraint graph** - nodes of graph correspond to vars, edge connects any two vars that participate in a constraint
- **precedence constraint** - define that a task with duration must occur and complete before the next: $T_1 + d_1 \leq T_2$
- **disjunctive constraint** - vars must not overlap in time; one must come first: $(AxleF + 10 \le AxleB) ;\text{or}; (AxleB + 10 \le AxleF)$
- Simplest CSP involve discrete and finite domains
- No algorithm exists for solving general nonlinear constraints on integer variables -- only linear ones
- **continuous domains** - example is linear programming -- mathematical method for determining optimal outcome such as max profit or lowest cost in a given model using linear relationships. Max or min a linear objective function subject to linear constraints
- **Unary constaint** - restricts value of a single var
- **Binary constraint** - relates two vars
- **Binary CSP** - one w/ only unary and binary constraints -- can be represetned as a constraint graph
- **ternary constaint** - involves 3
- **global constraint** - example is *Alldiff* whic hsays that all vars involved in constraint must have different values
    - need not involve all variables in a problem
- **constraint hypergraph** - ordinary nodes and hypernodes which represent n-ary constaints -- constraints involving n variables
- **preference constaints** - indicating which solutions are preferred
- **constrained optimization problem** - preference constraints encoded as costs on an individual variable assignments and cost against the overall objective function
    - linear program are one class of COPs

## Dual Graph Transformation

### Core Idea
- Original CSP:
  - Nodes = variables
  - Constraints connect variables
- Dual graph:
  - Nodes = **constraints**
  - Edges = **binary consistency constraints** between constraints that share variables
### Why do this?
- Convert an $n$-ary CSP into a **binary CSP**
- Allows use of binary CSP algorithms
- Simplifies algorithm design (conceptually)
### Original Example
Variables:
$$
X=\{X,Y,Z\}, \quad Dom=\{1,2,3,4,5\}
$$

Constraints:
$$
C_1:\; X+Y=Z
$$
$$
C_2:\; X+1=Y
$$
### Dual Graph Construction
#### Step 1 — Nodes
Create one dual variable per original constraint:
$$
X'=\{C_1,C_2\}
$$
#### Step 2 — Domains of dual variables
- Domain of each dual variable = satisfying tuples of the original constraint.
Example:
$$
Dom(C_1)=\{(x,y,z)\mid x+y=z\}
$$
Examples:
$$
(1,2,3), (2,3,5), \dots
$$
$$
Dom(C_2)=\{(x,y)\mid x+1=y\}
$$
Examples:
$$
(1,2), (2,3), \dots
$$
#### Step 3 — Binary constraint between dual nodes
- Add an edge between dual nodes if original constraints share variables.
- $C_1$ and $C_2$ share $X,Y$.

Define binary relation:
$$
R_1(C_1,C_2)
$$
Meaning:
- tuples must agree on shared variables.
Valid examples:
$$
((1,2,3),(1,2))
$$
$$
((2,3,5),(2,3))
$$

Invalid example:
$$
((1,2,3),(2,3))
$$
(mismatch in shared variables)
### Key Interpretation
Dual graph = selecting compatible tuples instead of assigning variable values.

Original view:
- assign values to variables

Dual view:
- select satisfying tuples
- enforce overlap consistency
### Graph Comparison
| Original (Primal) | Dual Graph |
|---|---|
| Node = variable | Node = constraint |
| Edge = constraint | Edge = binary consistency constraint |
| Values = single assignments | Values = tuples |
### Mental Model
- Each dual node = local mini-solution
- Edges ensure mini-solutions agree where they overlap
### Important Clarification
- Binary links are **constraints**, not variables.
- Dual variables are the original constraints.
### Why not always use dual graph?
1. Global constraints easier to write directly.
2. Domains can explode in size:
   - tuple count grows exponentially with constraint arity.
### Quick Rule
Two dual nodes are connected **iff** their original constraints share variables.
If constraints share no variables:
- no edge in dual graph.

## 6.2 Constraint Propagation: Inference in CSPs
- **Constraint propagation**: Process of using constraints to reduce the number of legal values for a variable, which can further reduce legal values for other variables.
- **Local consistency**: The key idea behind constraint propagation; treating variables as nodes in a graph and constraints as edges to eliminate inconsistent values throughout the graph.
- CSP algorithms can generate successors by choosing a new variable assignment or by performing constraint propagation as a preprocessing step or intertwined with search.
## 6.2.1 Node Consistency
- **Node-consistent**: A variable is node-consistent if all values in its domain satisfy the variable's unary constraints.
- A graph is **node-consistent** if every variable in the graph is node-consistent.
- Unary constraints can be eliminated at the start by reducing the domain of affected variables.
## 6.2.2 Arc Consistency
- **Arc-consistency**: A variable $X_i$ is arc-consistent with respect to another variable $X_j$ if for every value in the current domain $D_i$ there is some value in the domain $D_j$ that satisfies the binary constraint on the arc $(X_i, X_j)$. (arc is synonomous with edge)
- A graph is arc-consistent if every variable is arc-consistent with every other variable.
- **AC-3** algorithm: The most popular algorithm for enforcing arc consistency.
- AC-3 maintains a queue of arcs; it pops an arc $(X_i, X_j)$ and calls $REVISE(csp, X_i, X_j)$ to make $X_i$ arc-consistent with $X_j$.
- If $D_i$ is revised, all arcs $(X_k, X_i)$ where $X_k$ is a neighbor of $X_i$ (excluding $X_j$) are added back to the queue.
- Complexity: For $n$ variables, domain size $d$, and $c$ binary constraints, AC-3 runs in $O(cd^3)$ worst-case time.
~~~Python
function AC-3(csp) returns false if an inconsistency is found and true otherwise
  queue <- a queue of arcs, initially all the arcs in csp
  while queue is not empty do
    (Xi, Xj) <- POP(queue)
    if REVISE(csp, Xi, Xj) then
      if size of Di = 0 then return false
      for each Xk in Xi.NEIGHBORS - {Xj} do
        add (Xk, Xi) to queue
  return true
function REVISE(csp, Xi, Xj) returns true iff we revise the domain of Xi
  revised <- false
  for each x in Di do
    if no value y in Dj allows (x,y) to satisfy the constraint between Xi and Xj then
      delete x from Di; revised <- true
  return revised
~~~
## 6.2.3 Path Consistency
- Arc consistency cannot detect inconsistencies in problems like map coloring with only two colors for three mutually touching regions.
- **Path consistency**: Tightens binary constraints by using implicit constraints inferred by looking at triples of variables.
- A two-variable set $\{X_i, X_j\}$ is **path-consistent** with respect to a third variable $X_m$ if, for every assignment $\{X_i = a, X_j = b\}$ consistent with the constraints on $\{X_i, X_j\}$, there is an assignment to $X_m$ that satisfies the constraints on $\{X_i, X_m\}$ and $\{X_m, X_j\}$.
## 6.2.4 K-consistency
- **K-consistency**: A CSP is $k$-consistent if, for any set of $k-1$ variables and for any consistent assignment to those variables, a consistent value can always be assigned to any $k$-th variable.
- 1-consistency is node consistency; 2-consistency is arc consistency; 3-consistency is path consistency.
- **Strongly k-consistent**: A CSP that is $i$-consistent for all $i$ from 1 to $k$.
- If a CSP with $n$ nodes is **strongly n-consistent**, a solution can be found in $O(n^2d)$ time without backtracking.
- Establishing $k$-consistency takes time and space exponential in $k$.
### Meaning
Checks whether partial assignments can be extended. If not extendable → eliminate them.
### Levels
- **1-consistency:** remove values violating unary constraints.
- **2-consistency (arc):** remove values with no supporting neighbor values.
- **3-consistency:** remove value pairs that cannot extend to a third variable.
- **k-consistency:** remove tuples of size $k-1$ that cannot extend.
### Key Idea
Pre-prune future dead ends during search.
### One-line Summary
k-consistency = remove partial assignments that cannot lead to a full solution.
## 6.2.5 Global Constraints
- **Global constraint**: A constraint involving an arbitrary number of variables (not necessarily all variables).
- **Alldiff**: A common global constraint stating all involved variables must have distinct values.
- A simple inconsistency detection for **Alldiff**: If $m$ variables have $n$ possible distinct values and $m > n$, the constraint is violated.
- **Resource constraint** (**Atmost constraint**): Limits the sum or count of variables, such as total personnel assigned to tasks.
- **Bounds propagation**: Used for large integer domains; domains are represented by upper and lower bounds $[L, U]$.
- **Bounds-consistent**: A CSP is bounds-consistent if for every variable $X$, for both the lower and upper bounds of $D_X$, there exists a value for every other variable $Y$ satisfying the constraints.
## 6.2.6 Sudoku
- **Sudoku**: A CSP with 81 variables (one per square) and 27 **Alldiff** constraints for each unit (row, column, $3 \times 3$ box).
- Arc consistency (AC-3) can solve easy Sudoku puzzles by reducing domains to single values.
- **Naked triples**: A higher-order consistency strategy where three squares in a unit have domains containing the same three numbers (or subsets), allowing those numbers to be removed from other squares in that unit.
- This exemplifies the power of CSP: general mechanisms like arc and path consistency apply to any problem defined in the formalism.

## 6.3 Backtracking Search for CSPs
- **Backtracking search**: A form of depth-first search that chooses values for one variable at a time and backtracks when a variable has no legal values left to assign.
- **Commutativity**: CSPs are commutative because the order of application of any given set of actions (assignments) does not matter; the same partial assignment is reached regardless of the order.
- In a search tree for $n$ variables of domain size $d$, commutativity allows us to consider only a single variable at each node, reducing the number of leaves from $n! \cdot d^n$ to $d^n$.
- **Algorithm: BACKTRACKING-SEARCH**
~~~Python
function BACKTRACKING-SEARCH(csp) returns a solution or failure
    return BACKTRACK(csp, {})
function BACKTRACK(csp, assignment) returns a solution or failure
    if assignment is complete then
        return assignment
    var <- SELECT-UNASSIGNED-VARIABLE(csp, assignment)
    for each value in ORDER-DOMAIN-VALUES(csp, var, assignment) do
        if value is consistent with assignment then
            add {var = value} to assignment
            inferences <- INFERENCE(csp, var, assignment)
            if inferences != failure then
                add inferences to csp
                result <- BACKTRACK(csp, assignment)
                if result != failure then
                    return result
                remove inferences from csp
            remove {var = value} from assignment
    return failure
~~~
- **Backtracking Core Idea:** Backtracking keeps a current partial assignment and tries to extend it one variable at a time.
- **Node Meaning:** A node represents the current partial assignment, not a full copy of all variables or CSP state.
- **Process:** Choose an unassigned variable, try a value, and check consistency with prior assignments.
- **Failure:** If no value works for the next variable, undo the last assignment and try another value (backtrack).
- **Storage:** One evolving state is maintained; changes are undone rather than storing full states at each node.
- **One-Line Summary:** Backtracking incrementally builds assignments and reverses the last choice when no consistent extension exists.

## 6.3.1 Variable and Value Ordering
- **Minimum-remaining-values (MRV)**: Heuristic that selects the variable with the fewest "legal" values remaining; also called "most constrained variable" or "fail-first" heuristic.
- **Degree heuristic**: Tie-breaker for MRV that selects the variable involved in the largest number of constraints on other unassigned variables.
- **Least-constraining-value**: Heuristic for value ordering that prefers the value ruling out the fewest choices for neighboring variables in the constraint graph; a "fail-last" approach.
## 6.3.2 Interleaving Search and Inference
- **Forward checking**: Whenever a variable $X$ is assigned, it establishes arc consistency for it by deleting values from the domains of unassigned neighbors $Y$ that are inconsistent with $X$.
- **Maintaining Arc Consistency (MAC)**: A more powerful inference algorithm than forward checking; after variable $X_i$ is assigned, it calls AC-3 starting with arcs $(X_j, X_i)$ for all unassigned neighbors $X_j$.
## 6.3.3 Intelligent Backtracking: Looking Backward
- **Chronological backtracking**: The default policy of backing up to the most recent decision point when a search fails.
- **Conflict set**: For a variable $X$, the set of previous assignments that are in conflict with some value for $X$.
- **Backjumping**: Method that backtracks to the most recent assignment in the conflict set rather than the immediate predecessor.
- **Conflict-directed backjumping**: A deeper backjumping method where the conflict set for a variable $X_j$ includes the preceding variables that caused $X_j$ and subsequent variables to have no consistent solution.
- If variable $X_j$ fails with conflict set $conf(X_j)$, the algorithm backjumps to the most recent variable $X_i$ in the set and updates $X_i$'s conflict set: $conf(X_i) \leftarrow conf(X_i) \cup conf(X_j) - \{X_i\}$.
## 6.3.4 Constraint Learning
- **No-good**: A minimum set of variables and values from a conflict set that is responsible for an inconsistency.
- **Constraint learning**: The process of finding and recording no-goods as new constraints to avoid encountering the same contradictions later in the search.

## 6.4 Local Search for CSPs
- **Local search algorithms**: Use a **complete-state formulation** where every variable is assigned a value at all times.
- Search process involves changing the value of one variable at a time to resolve conflicts.
- **Min-conflicts heuristic**: A value selection strategy that chooses the value for a variable that results in the minimum number of conflicts with other variables.
- **Algorithm: MIN-CONFLICTS**
~~~Python
function MIN-CONFLICTS(csp, max_steps) returns a solution or failure
    inputs: csp, a constraint satisfaction problem
            max_steps, the number of steps allowed before giving up
    current <- an initial complete assignment for csp
    for i = 1 to max_steps do
        if current is a solution for csp then
            return current
        var <- a randomly chosen conflicted variable from csp.VARIABLES
        value <- the value v for var that minimizes CONFLICTS(csp, var, v, current)
        set var = value in current
    return failure
~~~
- **Performance**: Min-conflicts is effective for many CSPs; on the $n$-queens problem, run time is roughly independent of problem size (solves million-queens in average 50 steps).
- **Landscape**: The state-space landscape of a CSP under min-conflicts often contains a series of **plateaus**.
- **Plateau search**: Involves **sideways moves** to another state with the same score to help local search navigate off plateaus.
- **Tabu search**: A technique to avoid cycles and wandering by keeping a small list of recently visited states and forbidding the algorithm from returning to them.
- **Constraint weighting**: A method to concentrate search on important constraints by assigning a numeric weight to each constraint (initially all 1).
- At each step, the algorithm chooses a variable/value pair that results in the lowest total weight of all violated constraints.
- Weights are incremented for violated constraints: $w \leftarrow w + 1$ for each constraint violated by the current assignment.
- This adds topography to plateaus to ensure improvement is possible and incorporates learning about difficult constraints over time.
- **Online settings**: Local search is advantageous when a problem changes (e.g., airline scheduling repairs due to bad weather) because it can start from the current schedule and repair it with minimum changes.

## 6.5 The Structure of Problems
- **Constraint graph**: Visualization of a CSP where nodes represent variables and edges represent constraints.
- **Connected component**: A set of nodes in the constraint graph where a path exists between any two nodes; components that are not connected to each other represent independent subproblems.
- **Independent subproblems**: Decomposing a graph into components reduces complexity from $O(d^n)$ to $O(d^c \cdot n/c)$, where $c$ is the number of variables per subproblem.
- Example: Dividing a 100-variable Boolean CSP into four subproblems of 25 variables reduces solution time from the lifetime of the universe to less than a second.
- **Tree-structured CSP**: A constraint graph where any two variables are connected by only one path; can be solved in time linear in the number of variables.
- **Directional arc consistency (DAC)**: Under an ordering $X_1, X_2, \dots, X_n$, a CSP is DAC if every $X_i$ is arc-consistent with each $X_j$ for $j > i$.
- **Topological sort**: A linear ordering of nodes in a tree such that each variable appears after its parent.
- Tree-structured CSPs are solved in $O(nd^2)$ time.
- **Algorithm: TREE-CSP-SOLVER**
~~~Python
function TREE-CSP-SOLVER(csp) returns a solution, or failure
    inputs: csp, a CSP with components X, D, C
    n <- number of variables in X
    assignment <- an empty assignment
    root <- any variable in X
    X <- TOPOLOGICALSORT(X, root)
    for j = n down to 2 do
        MAKE-ARC-CONSISTENT(PARENT(Xj), Xj)
        if it cannot be made consistent then
            return failure
    for i = 1 to n do
        assignment[Xi] <- any consistent value from Di
        if there is no consistent value then
            return failure
    return assignment
~~~
## 6.5.1 Cutset Conditioning
- **Cutset conditioning**: Reducing a general constraint graph to a tree by assigning values to a specific subset of variables.
- **Cycle cutset**: A subset of variables $S$ such that the constraint graph becomes a tree after $S$ is removed.
- For each possible assignment to the variables in $S$, the algorithm removes inconsistent values from remaining variables and solves the resulting tree.
- Complexity: $O(d^c \cdot (n - c)d^2)$, where $c$ is the size of the cycle cutset.
- Example: If a 100-variable Boolean CSP has a cycle cutset of size $c=20$, it can be solved in a few minutes rather than the lifetime of the universe.

* **Cutset conditioning (idea):** Convert a hard CSP with cycles into an easy tree-structured CSP by assigning values to a small set of variables that break all cycles.
* **Why it works:** Tree-structured CSPs can be solved efficiently (linear-time propagation), while cycles cause combinatorial explosion.
* **Cycle cutset:** A subset of variables whose removal makes the constraint graph a tree or forest.
* **Key intuition:** Freeze a few highly connected variables → cycles disappear → solve the remaining structure easily.
* **What “removal” means:** Choose values for cutset variables and enforce constraints by deleting inconsistent values from neighboring domains; variables are not deleted from the original problem, only conditioned on.
* **Algorithm step 1:** Select a cycle cutset S so the remaining constraint graph becomes a tree.
* **Algorithm step 2:** For each assignment to S that satisfies internal constraints, prune inconsistent domain values from neighboring variables.
* **Algorithm step 3:** Solve the remaining tree CSP using the tree algorithm.
* **Algorithm step 4:** If a solution exists, combine it with the cutset assignment and return; otherwise try the next assignment.
* **Runtime intuition:** Total cost ≈ (number of assignments to cutset) × (cost of solving tree).
* **Complexity:** If cutset size = c and domain size = d, complexity is about d^c × O(n), which is far smaller than d^n when c is small.
* **Tradeoff:** You intentionally guess a small number of variables to avoid searching over many variables later.
* **Choosing a cutset:** Finding the smallest cutset is NP-hard, so practical methods use heuristics (e.g., high-connectivity variables).
* **Core mental model:** Strategic guessing that removes cycles and turns one hard problem into many easy ones.
* **Relation to later topics:** Same conditioning idea appears in probabilistic inference (e.g., Bayesian networks and junction-tree style reasoning).

## 6.5.2 Tree Decomposition
- **Tree decomposition**: Transforming a constraint graph into a tree where each node (meganode) represents a set of variables from the original problem.
- Three requirements for tree decomposition:
- 1. Every variable in the original problem must appear in at least one meganode.
- 2. Variables connected by a constraint must appear together in at least one meganode.
- 3. If a variable appears in two meganodes, it must appear in every meganode on the path between them.
- The meganodes are solved as a tree-structured CSP where the domain of each meganode is the set of consistent tuples for its constituent variables.
- **Tree width** ($w$): Defined as one less than the size of the largest meganode in the decomposition; the tree width of a graph is the minimum width among all possible decompositions.
- Complexity: $O(nd^{w+1})$ where $w$ is the tree width.
- Comparison: Tree decomposition is faster than cutset conditioning if $w < c+1$, but requires exponential memory $O(d^w)$.
## 6.5.3 Value Symmetry
- **Value symmetry**: Redundancy in the search space caused by variables having interchangeable values (e.g., permuting color names in map coloring).
- For $d$ colors, there are $d!$ redundant versions of every solution.
- **Symmetry-breaking constraint**: An additional constraint used to eliminate redundant solutions by imposing an arbitrary order on values (e.g., $NT < SA < WA$).